# Install Libraries

In [1]:
!pip install "python-doctr[torch]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 41.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 87.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.9/288.9 kB 20.4 MB/s eta 0:00:00
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=c3281846754b90760b799b0d76f6e6ee49b1f16e570c7708f0c9f748309d804c
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e

In [ ]:
!pip install tesseract

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 MB 23.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for tesseract: filename=tesseract-0.1.3-py3-none-any.whl size=45562552 sha256=2547301f517c06261a80957445f63d85a70fbfbc263de902a8430f019c2979af
  Stored in directory: /root/.cache/pip/wheels/13/1f/8e/2d6c0e358fd6d01ca80ecd9185a374bcda35879f4fec727242
Successfully built tesseract


In [ ]:
!pip install pytesseract

In [ ]:
!pip install easyocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 19.2 MB/s eta 0:00:00


#Import Libraries

In [2]:
import cv2
# import pytesseract as pyt
# import easyocr
import matplotlib.pyplot as plt
# from PIL import Image
import os
import numpy as np
from doctr.models import ocr_predictor
from doctr.io import DocumentFile

#modling

In [3]:
def preprocess_variants(gray):
    variants = []
    #Raw resized
    raw = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    variants.append(raw)
    #CLAHE only
    clahe = cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
    v2 = clahe.apply(raw)
    variants.append(v2)
    #Adaptive mean threshold
    v3=cv2.adaptiveThreshold(raw,127,cv2.ADAPTIVE_THRESH_MEAN_C,cv2.THRESH_BINARY,9,2)
    variants.append(v3)
    #bitwise_not
    v4=cv2.bitwise_not(raw)
    return variants


In [ ]:
def ocr_tesseract(image):
    config = r'--oem 3 --psm 6 -l eng'
    pil = Image.fromarray(image)
    return pyt.image_to_string(pil, config=config)


In [ ]:
reader = easyocr.Reader(['en'], gpu=False)

def ocr_easy(image):
    result = reader.readtext(image, detail=0)
    return " ".join(result)


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

In [ ]:
def score_text(text):
    if not text:
        return 0
    return sum(c.isalnum() for c in text)


In [ ]:
def robust_ocr(image_path):
    gray = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    variants = preprocess_variants(gray)

    best_text = ""
    best_score = 0
    best_model=''
    for img in variants:
        # Tesseract
        t_text = ocr_tesseract(img)
        t_score = score_text(t_text)

        if t_score > best_score:
            best_score = t_score
            best_text = t_text
            best_model='tesseract'

        # EasyOCR
        e_text = ocr_easy(img)
        e_score = score_text(e_text)

        if e_score > best_score:
            best_score = e_score
            best_text = e_text
            best_model='easy_ocr'

    print(best_model)
    print("\n")
    return best_text


In [ ]:
folder = "/content/drive/MyDrive/Graduation/samples"

for file in os.listdir(folder):
    if file.lower().endswith((".jpg", ".png", ".jpeg")):
        path = os.path.join(folder, file)
        print(f"\n{file}")
        print(robust_ocr(path))



sample_2.jpeg
easy_ocr


Purified Water T Triethanolamine a Petrolatum Cocos Nucifera Oil (Coconut Oil) Paraffinum Liquidum Argania Spinosa Kernel Oil (Argan Oil) Butyrospermum Parkii Butter (Shea Butter) Cyclopentasiloxane  Dimethicone Ie Beeswax - Stearic acid L Cetyl alcohol K Microcrystalline wax - Methylparaben 1 Parfume_ Ingredients

sample_1.jpeg
easy_ocr


WGredientBQuA-GLCENiN; GLYCERYLSTEARAEE SE,PARAFFINUMLIQUIDUMILOHYLHEXYESTEARATEZ HMDROGENATEDGOGO GYyLeRIDES CETEARYL ALcoHoLDIETHIGONBCRAMOMILAREcUTIA GLOWEREXIRAGEDISADOLOL GLuCOSE TOCOPHERLL ACETATEILAGTICACID GZCIOPENTASILOXANER GYCLOHEXASLOXANE SELERCIC ACID PALMITICACIDB CROPYLENE GLYCOL KANTLANGU  CARBOMERS SODIUMHYDROXIDE IYLENE GLYCOL MethyoarabenCropyLo  ABEN [DAzOLDI UREA SODuUnBENZOAIESROTASSLUMSORBATB

sample_3.jpeg
easy_ocr


Damowhite" WF € Kojiz acid vitamin B3(NIACINAMDE} Glycokc tid Punthenol, Sodium IaJreth sulfate, Cocamidopropyl betoie Eommide DEA Ascorbic acid, Styrenelacrylates copolymer; Glabra Root 

In [4]:
# class OCR functions
class OCR_test:
  def __init__(self,image_path) -> None:
     self.__image_path=image_path
     #Initalize model
     self.doctr_model = ocr_predictor(det_arch='db_resnet50', reco_arch='crnn_vgg16_bn', pretrained=True)

  def __ocr_doctr(self,image_rgb)->str:
    #Initalize model with detection and recognition stage
    doctr_model = ocr_predictor(det_arch='db_resnet50', reco_arch='crnn_vgg16_bn', pretrained=True)
    try:
        # temp path to save numpy array and avoid type error conflict
        temp_path = "doctr_temp_file.jpg"
        cv2.imwrite(temp_path, cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR))
        # Standraize and normalize images
        doc = DocumentFile.from_images([temp_path])
        #Inference
        result = doctr_model(doc)
        #Parsing the export
        export = result.export()
        #The reconstraction
        full_text = []
        for page in export['pages']:
            for block in page['blocks']:
                for line in block['lines']:
                    line_text = " ".join([word['value'] for word in line['words']])
                    full_text.append(line_text)
        if os.path.exists(temp_path):
            os.remove(temp_path)

        return " ".join(full_text)
    except Exception as e:
        print(f"DocTR Error: {e}")
        return ""

  def __normalize_text(self,text):
    text=text.lower()
    return text

  def robust_ocr(self)->str:
        img_bgr = cv2.imread(self.__image_path)

        if img_bgr is None:
            return "Error: Image not found"

        raw_rgb = cv2.resize(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB),
                             None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)

        d_text = self.__ocr_doctr(raw_rgb)
        d_text = self.__normalize_text(d_text)
        return d_text

In [5]:
folder = "/content/drive/MyDrive/Graduation/samples"

for file in os.listdir(folder):
    if file.lower().endswith((".jpg", ".png", ".jpeg")):
        path = os.path.join(folder, file)
        print(f"\n{file}")
        ocr_object=OCR_test(path)
        text=ocr_object.robust_ocr()
        print(text)




sample_2.jpeg


  0%|          | 0/102021912 [00:00<?, ?it/s]

  0%|          | 0/63303144 [00:00<?, ?it/s]

error: OpenCV(4.13.0) /io/opencv/modules/imgproc/src/clahe.cpp:360: error: (-215:Assertion failed) _src.type() == CV_8UC1 || _src.type() == CV_16UC1 in function 'apply'


In [ ]:
def __ocr_doctr(image_rgb)->str:
    #Initalize model with detection and recognition stage
    doctr_model = ocr_predictor(det_arch='db_resnet50', reco_arch='crnn_vgg16_bn', pretrained=True)
    try:
        # temp path to save numpy array and avoid type error conflict
        temp_path = "doctr_temp_file.jpg"
        cv2.imwrite(temp_path, cv2.cvtColor(image_rgb, cv2.COLOR_2BGR))
        # Standraize and normalize images
        doc = DocumentFile.from_images([temp_path])
        #Inference
        result = doctr_model(doc)
        #Parsing the export
        export = result.export()
        #The reconstraction
        full_text = []
        for page in export['pages']:
            for block in page['blocks']:
                for line in block['lines']:
                    line_text = " ".join([word['value'] for word in line['words']])
                    full_text.append(line_text)
        if os.path.exists(temp_path):
            os.remove(temp_path)

        return " ".join(full_text)

In [8]:
def robust_ocr_doc(image_path):
    gray = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    variants = preprocess_variants(gray)
    texts=[]
    i=1
    for img in variants:
       doctr_model = ocr_predictor(det_arch='db_resnet50', reco_arch='crnn_vgg16_bn', pretrained=True)
       # temp path to save numpy array and avoid type error conflict
       temp_path = "doctr_temp_file.jpg"
       cv2.imwrite(temp_path,img)
       # Standraize and normalize images
       doc = DocumentFile.from_images([temp_path])
       #Inference
       result = doctr_model(doc)
       #Parsing the export
       export = result.export()
       #The reconstraction
       full_text = []
       for page in export['pages']:
          for block in page['blocks']:
              for line in block['lines']:
                  line_text = " ".join([word['value'] for word in line['words']])
                  full_text.append(line_text)
       if os.path.exists(temp_path):
          os.remove(temp_path)
       texts.append("\n")
       texts.append(f'version: {i}')
       texts.append(" ".join(full_text))
       i+=1
    return texts


folder = "/content/drive/MyDrive/Graduation/samples"

for file in os.listdir(folder):
    if file.lower().endswith((".jpg", ".png", ".jpeg")):
        path = os.path.join(folder, file)
        print(f"\n{file}")
        print(robust_ocr_doc(path))




sample_2.jpeg
['\n', 'version: 1', 'Ingredients Purified Water - Triethanolamine - Petrolatum Cocos Nucifera Oil (Coconut Oil)- Paraffinum Liquidum - Argania Spinosa Kernel Oil (Argan Oil) - Butyrospermum Parkii Butter (Shea Butter) Cyclopentasiloxane - Dimethicone - Beeswax Stearic acid - - Cetyl alcohol - - Microcrystalline wax Methylparaben - Parfume.', '\n', 'version: 2', 'Ingredients Purified Water - Triethanolamine - Petrolatum Cocos Nucifera Oil (Coconut Oil)- Paraffinum Liquidum - Argania Spinosa Kernel Oil (Argan Oil) a a Butyrospermum Parkii Butter (Shea Butter) Cyclopentasiloxane - Dimethicone - Beeswax Stearic acid - - Cetyl alcohol - a Microcrystalline wax Methylparaben - Parfume.', '\n', 'version: 3', 'ingredients Purified Water : i Triethanolamine - Petrolatum Cocos Nucifera Oil(Coconut Oil) a Paraffinum Liquidume Argania SpinosaKeme/Ol(Agon Oil) Butyrospermum Parkii Butter (Shea Butter). Cyclopentasiloxane - = Dimethicone - Beeswax : Steariciacid F a Cetylalcohol Micro